# ⚡ Módulo 12 - Notebook 03: Joins Distribuidos y Broadcast

## 🔗 Uniones a gran escala con optimización

**Libro:** Saliendo de lo Pandito  
**Módulo:** 12 - PySpark Transformación Avanzada  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Ejecutar** joins distribuidos a gran escala  
✅ **Optimizar** con Broadcast Joins  
✅ **Entender** shuffle y su impacto en performance  
✅ **Elegir** el tipo de join correcto  
✅ **Evitar** skew de datos en joins

---

## 📋 Pre-requisitos

* ✅ Notebooks 12_01 y 12_02 completados
* ✅ Conocimiento de joins en Pandas
* ✅ Familiaridad con transformaciones Spark

---

## 📚 Contenido

1. Joins en Spark: Tipos y Sintaxis
2. Shuffle: El Costo de los Joins
3. Broadcast Joins: Optimización
4. Join Skew y Cómo Evitarlo
5. Best Practices para Joins
6. Caso Integrador: Join de Millones de Registros

---

## 💡 Por qué importa

**Joins son la operación más costosa en Spark:**

* 🔄 **Shuffle:** Movimiento de datos entre nodos
* 📊 **Broadcast:** Optimización 10-100x más rápida
* ⚠️ **Skew:** Desbalanceo que paraliza clusters
* ⚡ **Optimización:** Diferencia entre minutos y horas

**Dominar joins = dominar Spark**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (tabla grande)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    # Crear tabla de sucursales (tabla pequeña - candidata a broadcast)
    df_sucursales = df_ventas.select(
        "sucursal_id",
        "sucursal_nombre",
        "zona",
        "lat",
        "lon"
    ).distinct()
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"\n📊 Tabla GRANDE (ventas):")
    print(f"   Registros: {df_ventas.count():,}")
    print(f"   Particiones: {df_ventas.rdd.getNumPartitions()}")
    
    print(f"\n🏪 Tabla PEQUEÑA (sucursales):")
    print(f"   Registros: {df_sucursales.count():,}")
    print(f"   Particiones: {df_sucursales.rdd.getNumPartitions()}")
    
    print(f"\n🎯 Escenario perfecto para Broadcast Join:")
    print(f"   • Tabla grande (ventas): Miles de registros")
    print(f"   • Tabla pequeña (sucursales): Decenas de registros")
    print(f"   • Join key: sucursal_id")
    
    print(f"\n🔗 Este notebook demostrará:")
    print(f"   • Join normal vs Broadcast Join")
    print(f"   • Impacto del shuffle en performance")
    print(f"   • Cuándo usar cada tipo de join")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Joins Distribuidos: Performance Crítico

### 🔗 Tipos de Joins en Spark

**Sintaxis:**
```python
df1.join(df2, on="clave", how="tipo")
```

**Tipos disponibles:**

| Tipo | Descripción | Uso |
|------|-------------|-----|
| **inner** | Solo coincidencias | Default, más común |
| **left** (left_outer) | Todas de izq + coincidencias | Mantener todas las ventas |
| **right** (right_outer) | Todas de der + coincidencias | Mantener todos los clientes |
| **outer** (full_outer) | Todas de ambas | Conciliación completa |
| **left_semi** | Filtro (solo izq que coinciden) | EXISTS en SQL |
| **left_anti** | Filtro (solo izq que NO coinciden) | NOT EXISTS en SQL |

---

### 🔄 Shuffle: El Costo Oculto

**¿Qué es el shuffle?**

Movimiento de datos entre particiones para que las claves coincidentes estén en el mismo nodo.

**Proceso de join normal (Sort-Merge Join):**

```
1. Hash de la clave de join
2. Redistribuir datos por hash (SHUFFLE)
3. Ordenar cada partición
4. Merge de particiones ordenadas
```

**Costo del shuffle:**
* 💾 Escritura a disco
* 🌐 Transferencia de red
* 🔄 Serialización/deserialización
* ⏱️ **Puede ser 10-100x más lento**

---

### 📡 Broadcast Join: La Optimización

**Concepto:**
Enviar la tabla PEQUEÑA a TODOS los executors (evita shuffle).

**Cuándo usar:**
* Tabla pequeña < 10 MB (configurable)
* Join de fact table con dimension table
* Tablas de lookup/referencia

**Sintaxis:**
```python
from pyspark.sql.functions import broadcast

# Método 1: Explícito
df_result = df_grande.join(
    broadcast(df_pequeno),
    on="clave",
    how="inner"
)

# Método 2: Configuración automática
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")  # 10 MB
```

**Ventajas:**
* ⚡ **10-100x más rápido**
* 🚫 **Sin shuffle**
* 📉 **Menos memoria en executors**

**Desventajas:**
* ⚠️ Solo para tablas pequeñas (< memoria driver)
* 💾 Usa memoria del driver

---

### ⚠️ Join Skew: El Asesino de Performance

**¿Qué es skew?**

Cuando una clave tiene MUCHOS más valores que otras.

**Ejemplo:**
```
Clave     Registros
-----     ---------
A         10
B         15
C         9,999,900  ← SKEW!
D         12
```

**Problema:**
* Una partición procesa 99.9% de los datos
* Las demás esperan ociosas
* Job lentísimo

**Solución: Salting**
```python
# Añadir "sal" aleatoria a la clave
df = df.withColumn(
    "clave_salted",
    F.concat(F.col("clave"), F.lit("_"), (F.rand() * 10).cast("int"))
)

# Join por clave salted
```

---

### 🎯 Best Practices para Joins

**1️⃣ Usar Broadcast cuando sea posible**
```python
# Tabla pequeña < 10 MB
df.join(broadcast(df_pequeno), "clave")
```

**2️⃣ Filtrar ANTES de join**
```python
# ❌ MAL
df1.join(df2, "clave").filter("fecha > '2023-01-01'")

# ✅ BIEN
df1_filtrado = df1.filter("fecha > '2023-01-01'")
df1_filtrado.join(df2, "clave")
```

**3️⃣ Repartir antes de join (si ambas son grandes)**
```python
df1.repartition("clave").join(df2.repartition("clave"), "clave")
```

**4️⃣ Evitar múltiples joins consecutivos**
```python
# ❌ MAL (3 shuffles)
df1.join(df2, "a").join(df3, "b").join(df4, "c")

# ✅ BIEN (1 shuffle si usas broadcast)
df1.join(broadcast(df2), "a") \
   .join(broadcast(df3), "b") \
   .join(broadcast(df4), "c")
```

---

### 📊 Comparativa: Join Normal vs Broadcast

**Escenario:**
* df_ventas: 10 millones de registros
* df_sucursales: 50 registros

**Join normal (Sort-Merge):**
```python
df_ventas.join(df_sucursales, "sucursal_id")  # ~5 minutos
```

**Broadcast Join:**
```python
df_ventas.join(broadcast(df_sucursales), "sucursal_id")  # ~5 segundos
```

**Mejora: 60x más rápido** ⚡

---

### 💼 Caso de Uso: Enriquecer Ventas con Info de Sucursal

```python
from pyspark.sql.functions import broadcast

# Tabla grande: millones de transacciones
df_transacciones = spark.table("ventas.transacciones")

# Tabla pequeña: decenas de sucursales
df_sucursales = spark.table("maestros.sucursales")

# Broadcast join (optimal)
df_enriquecido = df_transacciones.join(
    broadcast(df_sucursales),
    on="sucursal_id",
    how="inner"
)

# Ahora tenemos: transacción + nombre_sucursal + zona + gerente
df_enriquecido.select(
    "fecha",
    "monto",
    "nombre_sucursal",
    "zona",
    "gerente"
).show()
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("🔗 JOINS DISTRIBUIDOS Y BROADCAST")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
    
    # Mostrar configuración de broadcast
    threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
    print(f"\nBroadcast Join Threshold: {threshold}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Tipos de joins (inner, left, outer, semi, anti)")
print("  • Shuffle y su impacto en performance")
print("  • Broadcast joins para optimización")
print("  • Join skew y cómo evitarlo")

print("\n📖 Métodos clave:")
print("  - df1.join(df2, on='key', how='inner')")
print("  - df1.join(broadcast(df2), on='key')")
print("  - df.repartition('key')  # Pre-shuffle")
print("  - spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10m')")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🔗 Joins con datos reales de Los Andes Market

### 📊 Escenario real de join

El notebook ya carga dos tablas:
* `df_ventas` (tabla grande): miles de registros mensuales por sucursal
* `df_sucursales` (tabla pequeña): catálogo de sucursales con coordenadas

Este es el escenario clásico de **Broadcast Join**: tabla grande + tabla pequeña.

```python
from pyspark.sql.functions import broadcast

# Broadcast: envía la tabla pequeña a todos los executors
df_result = df_ventas.join(broadcast(df_sucursales), on="sucursal_id", how="inner")
```

---

### 🔄 Tipos de join con datos reales

| Tipo | Caso de uso en Los Andes Market |
|------|--------------------------------|
| `inner` | Solo ventas con sucursal válida |
| `left` | Todas las ventas (incluso sin sucursal) |
| `left_semi` | Ventas que tienen sucursal (EXISTS) |
| `left_anti` | Ventas huérfanas (sin sucursal) |

---

### 💡 Preguntas de negocio
* ¿Todas las ventas tienen sucursal válida? (inner vs left)
* ¿Hay ventas sin sucursal registrada? (left_anti)
* ¿El broadcast join mejora el rendimiento vs sort-merge?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time

print("🔗 JOINS CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None and df_sucursales is not None:
    print("\n1️⃣  INNER JOIN: Ventas + sucursales")
    print("-"*70)

    df_inner = df_ventas.join(df_sucursales, on="sucursal_id", how="inner")
    print(f"\n   Registros después de inner join: {df_inner.count():,}")
    df_inner.select(
        "sucursal_id", "sucursal_nombre", "zona", "ventas", "fecha", "lat", "lon"
    ).show(5, truncate=30)

    print("\n" + "="*70)
    print("\n2️⃣  BROADCAST JOIN: Optimización")
    print("-"*70)

    # Join normal (sort-merge)
    start = time.time()
    df_normal = df_ventas.join(df_sucursales, on="sucursal_id", how="inner")
    n_normal = df_normal.count()
    t_normal = time.time() - start

    # Broadcast join (envía tabla pequeña a todos los executors)
    start = time.time()
    df_broadcast = df_ventas.join(broadcast(df_sucursales), on="sucursal_id", how="inner")
    n_broadcast = df_broadcast.count()
    t_broadcast = time.time() - start

    print(f"\n   Join normal (sort-merge): {n_normal:,} filas en {t_normal:.3f}s")
    print(f"   Broadcast join:           {n_broadcast:,} filas en {t_broadcast:.3f}s")
    print(f"\n   💡 En datasets pequeños la diferencia es mínima")
    print(f"   💡 En GB/TB el broadcast puede ser 10-100x más rápido")

    print("\n" + "="*70)
    print("\n3️⃣  LEFT JOIN: Mantener todas las ventas")
    print("-"*70)

    df_left = df_ventas.join(df_sucursales, on="sucursal_id", how="left")
    n_left = df_left.count()
    n_nulls = df_left.filter(F.col("sucursal_nombre").isNull()).count()
    print(f"\n   Registros con left join: {n_left:,}")
    print(f"   Ventas sin sucursal (null): {n_nulls}")
    if n_nulls > 0:
        print("   ⚠️  Hay ventas huérfanas (sin sucursal registrada)")
    else:
        print("   ✅ Todas las ventas tienen sucursal válida")

    print("\n" + "="*70)
    print("\n4️⃣  LEFT_SEMI: Ventas que tienen sucursal (EXISTS)")
    print("-"*70)

    df_semi = df_ventas.join(df_sucursales, on="sucursal_id", how="left_semi")
    print(f"\n   left_semi: {df_semi.count():,} ventas con sucursal existente")
    print("   💡 left_semi solo devuelve columnas de df_ventas (no duplica)")
    df_semi.select("sucursal_id", "ventas", "fecha").show(5, truncate=30)

    print("\n" + "="*70)
    print("\n5️⃣  LEFT_ANTI: Ventas sin sucursal (NOT EXISTS)")
    print("-"*70)

    df_anti = df_ventas.join(df_sucursales, on="sucursal_id", how="left_anti")
    n_anti = df_anti.count()
    print(f"\n   left_anti: {n_anti} ventas sin sucursal registrada")
    if n_anti > 0:
        print("   ⚠️  Hay ventas huérfanas — revisar calidad de datos:")
        df_anti.select("sucursal_id", "ventas", "fecha").show(5, truncate=30)
    else:
        print("   ✅ No hay ventas huérfanas — integridad referencial OK")

    print("\n" + "="*70)
    print("\n6️⃣  JOIN CON AGREGACIÓN: Pipeline completo")
    print("-"*70)

    # Join ventas + sucursales y luego agregar por zona
    pipeline = (df_ventas
        .join(broadcast(df_sucursales), on="sucursal_id", how="inner")
        .groupBy("zona")
        .agg(
            F.round(F.sum("ventas"), 0).alias("ventas_totales"),
            F.countDistinct("sucursal_id").alias("num_sucursales"),
            F.round(F.avg("ventas"), 0).alias("ventas_promedio"),
            F.round(F.max("ventas"), 0).alias("venta_max")
        )
        .orderBy(F.desc("ventas_totales"))
    )
    print("\n   Pipeline: join + groupBy + agg por zona:")
    pipeline.show(truncate=30)

    print("\n" + "="*70)
    print("\n7️⃣  EXPLAIN: Ver plan de ejecución")
    print("-"*70)

    print("\n   Plan del broadcast join:")
    df_broadcast.explain()
    print("\n   💡 'BroadcastExchange' indica que Spark usó broadcast (sin shuffle)")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del Notebook 12_03

### ✅ Lo que aprendiste

1. **Tipos de joins en Spark:**
   ```python
   df1.join(df2, on="clave", how="inner")   # Solo coincidencias
   df1.join(df2, on="clave", how="left")    # Todas de izq
   df1.join(df2, on="clave", how="outer")   # Todas de ambas
   df1.join(df2, on="clave", how="left_semi")  # EXISTS
   df1.join(df2, on="clave", how="left_anti")   # NOT EXISTS
   ```

2. **Shuffle: el costo oculto de los joins:**
   ```python
   # Sort-Merge Join (default para tablas grandes):
   # 1. Hash de la clave de join
   # 2. Redistribuir datos por hash (SHUFFLE)
   # 3. Ordenar cada partición
   # 4. Merge de particiones ordenadas
   # 
   # Costo: escritura a disco + transferencia de red + serialización
   # Puede ser 10-100x más lento que broadcast
   ```

3. **Broadcast Join: la optimización 10-100x:**
   ```python
   from pyspark.sql.functions import broadcast
   
   # Envía la tabla pequeña a TODOS los executors (sin shuffle)
   df_result = df_grande.join(
       broadcast(df_pequeno),
       on="clave",
       how="inner"
   )
   
   # Auto-broadcast (configurable)
   spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")
   ```

4. **Join Skew y salting:**
   ```python
   # Skew: una clave tiene 99.9% de los registros
   # Solución: añadir "sal" aleatoria a la clave
   df = df.withColumn(
       "clave_salted",
       F.concat(F.col("clave"), F.lit("_"), (F.rand() * 10).cast("int"))
   )
   # Join por clave salted distribuye la carga
   ```

5. **Best practices para joins:**
   - Filtrar ANTES de join (reduce datos a shufflear)
   - Broadcast para tablas < 10 MB
   - Repartition por clave si ambas son grandes
   - Evitar múltiples joins consecutivos sin broadcast

---

### 🛠️ Guía rápida de joins en Spark

**Caso 1: Broadcast join (tabla grande + tabla pequeña)**
```python
from pyspark.sql.functions import broadcast
df_result = df_ventas.join(broadcast(df_sucursales), on="sucursal_id")
# Sin shuffle, 10-100x más rápido
```

**Caso 2: Join normal (ambas tablas grandes)**
```python
df_result = df1.join(df2, on="cliente_id", how="inner")
# Sort-Merge Join con shuffle
```

**Caso 3: left_semi y left_anti (filtros por existencia)**
```python
# left_semi = EXISTS (solo filas de izq que coinciden)
df_clientes.join(df_compras, on="cliente_id", how="left_semi")

# left_anti = NOT EXISTS (solo filas de izq que NO coinciden)
df_clientes.join(df_compras, on="cliente_id", how="left_anti")
```

**Caso 4: Salting para skew extremo**
```python
df_salted = df.withColumn("key_salted",
    F.concat(F.col("key"), F.lit("_"), (F.rand() * 10).cast("int")))
df_result = df_salted.join(df2_salted, on="key_salted")
```

**Caso 5: Filtrar antes de join**
```python
# MALO: join primero, filtra después (shufflea TODO)
df1.join(df2, "clave").filter("fecha > '2024-01-01'")

# BUENO: filtra primero, join después (shufflea menos)
df1.filter("fecha > '2024-01-01'").join(df2, "clave")
```

---

### 🏆 Resumen del Módulo 12

**Aprendiste:**

1. **12_01 - PySpark Transformación Avanzada:** withColumn, F.functions, when/otherwise, nulos, casts
2. **12_02 - Operaciones Columnas y Functions:** filter, select, groupBy, window functions
3. **12_03 - Joins Distribuidos y Broadcast:** Tipos de join, shuffle, broadcast, skew, salting

**Habilidades adquiridas:**
* ✅ Ejecutar joins distribuidos a gran escala
* ✅ Optimizar con Broadcast Joins (10-100x más rápido)
* ✅ Entender y mitigar el costo del shuffle
* ✅ Detectar y resolver Join Skew con salting
* ✅ Aplicar best practices para pipelines ETL con Spark

---


<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔗 ¡Módulo 12 Completado!</h3>
  <p><i>"Dominas PySpark Transformación Avanzada: withColumn, functions, window y joins distribuidos. Ahora puedes construir ETL que escala de MB a TB sin perder performance."</i></p>
</div>